# 状態異常・HP・能力ランクを直接操作したシナリオ間で致死率を比較する

02（各操作の単体確認）・03（calc_lethal 基本）を踏まえ、battle.set_ailment() /
modify_hp() / modify_stats() / faint() で「対戦を進行させずに特定の状態から
始まるシナリオ」を組み立て、calc_lethal() の結果がどう変わるかを比較する。

Google Colabで開いた場合は、まず次のセルで `jpoke` をインストールする
（ローカルで `pip install jpoke` 済みなら不要）。

In [ ]:
%pip install -q jpoke

In [ ]:
from __future__ import annotations

from jpoke import Battle, Player

In [ ]:
move_name = "ドラゴンテール"

# calc_lethal()が状態異常・天候ダメージを自動合算する点は
# docs/api/README.md の calc_lethal() 節を参照。努力値無振りのペアで比較する
print("努力値無振りの攻撃側で、どく無し/どくありを比較:")

plain_attacker_player = Player("PlainAttacker")
plain_attacker_player.add_pokemon("ガブリアス", move_names=[move_name])
plain_defender_player = Player("PlainDefender")
plain_defender_player.add_pokemon("カイリュー", item_name="オボンのみ")

plain_battle = Battle(plain_attacker_player, plain_defender_player, seed=1)
plain_battle.start()
plain_attacker = plain_battle.get_active(plain_attacker_player)
plain_defender = plain_battle.get_active(plain_defender_player)

no_ailment_final = plain_battle.calc_lethal(
    attacker=plain_attacker, moves=move_name, max_attack=2,
)[-1]

In [ ]:
# set_ailment()/status/has_ailment()の詳細は docs/api/README.md を参照
plain_battle.set_ailment(plain_defender, "どく")
print(
    f"防御側の状態異常: {plain_defender.status}"
    f"（has_ailment('どく') = {plain_defender.has_ailment('どく')}）"
)
poisoned_final = plain_battle.calc_lethal(
    attacker=plain_attacker, moves=move_name, max_attack=2,
)[-1]

print(
    f"{move_name}を{no_ailment_final.n_attack}発当てた時点の致死率: "
    f"どく無し {no_ailment_final.lethal_probability:.2%} / "
    f"どくあり {poisoned_final.lethal_probability:.2%}"
)

In [ ]:
# modify_hp()/modify_stats() で「特定のHP・ランク状態から始まる」シナリオも
# 直接組み立てられる（Pokemon.hpへの直接代入を避けるべき理由は
# docs/api/README.md の Battle「シナリオ構築系」節を参照）
print("-" * 50)
print("防御側のHP・ランク状態を変えて比較:")

scenario_attacker_player = Player("ScenarioAttacker")
scenario_attacker_player.add_pokemon("ガブリアス", move_names=[move_name])
scenario_defender_player = Player("ScenarioDefender")
scenario_defender_player.add_pokemon("カイリュー")

scenario_battle = Battle(scenario_attacker_player, scenario_defender_player, seed=1)
scenario_battle.start()
scenario_attacker = scenario_battle.get_active(scenario_attacker_player)
scenario_defender = scenario_battle.get_active(scenario_defender_player)

# max_attack=1 に絞り、HP・ランク状態を変えた結果が1発の致死率にそのまま反映されるようにする
full_hp_final = scenario_battle.calc_lethal(
    attacker=scenario_attacker, moves=move_name, max_attack=1,
)[-1]

In [ ]:
# modify_hp(r=-0.6) で最大HPの60%分のダメージを与え、残りHP40%の状態を直接作る
# （引数の詳細は docs/api/README.md の modify_hp() 節を参照）
scenario_battle.modify_hp(scenario_defender, r=-0.6, reason="シナリオ構築用")
damaged_final = scenario_battle.calc_lethal(
    attacker=scenario_attacker, moves=move_name, max_attack=1,
)[-1]
print(
    f"{move_name}を1発当てた時点の致死率: "
    f"HP満タン {full_hp_final.lethal_probability:.2%} / "
    f"HP{scenario_defender.hp}/{scenario_defender.max_hp} "
    f"{damaged_final.lethal_probability:.2%}"
)

In [ ]:
# modify_stats() は複数の能力ランクを同時に変更できる
scenario_battle.modify_stats(scenario_defender, {"def": 1})
raised_def_final = scenario_battle.calc_lethal(
    attacker=scenario_attacker, moves=move_name, max_attack=1,
)[-1]
print(f"上記の状態からさらにぼうぎょ+1した後の致死率: {raised_def_final.lethal_probability:.2%}")

In [ ]:
# faint()の詳細は docs/api/README.md の Battle「シナリオ構築系」節を参照
scenario_battle.faint(scenario_defender)
print(f"faint()呼び出し後: fainted={scenario_defender.fainted}")

試してみよう: modify_hp(r=-0.6) の割合や set_ailment() の状態異常を変えると、
致死率がどう変わるか比較できる